## 获取任务sql

In [1]:
from voydstools.common.DataServiceAPI import DataServiceHttp

In [2]:
# For test
x_app_key = '50214295'
secret_key = 'e467683796fe3ae3792bf83fa0ef2796bb47dd90'


def get_sql_from_task_id(project_id, task_id):
    api = DataServiceHttp(x_app_key, secret_key)
    api.set_api("get_tabel_sql_temp_sql_df")
    query_data = {
        "dt": "2024-09-13",
        "project_id": project_id,
        "task_id": task_id
    }
    response = api.post(query_data)
    return response[0]['file_content']


In [3]:
sql = get_sql_from_task_id('3100', '31041901')
sql

请求：http://10.88.128.15:8000/dataservice/gateway/v1/api/get_tabel_sql_temp_sql_df，数据：{'dt': '2024-09-13', 'project_id': '3100', 'task_id': '31041901', 'pageSize': 5000, 'page': 1}


"--@exclude_dependency=sparklingwater.gateway_daily_hudi_ods_brp_--SPARK_SQL_brp_--********************************************************************--_brp_--author:zhengzong_brp_--create time:2024-08-01 16:26:40_brp_--desc:雨刷器档位_brp_--remind:请在资源引用中添加需要引用的资源_brp_--********************************************************************--_brp_--- DROP TABLE IF EXISTS app_trip_rt_wipe;_brp_-- create table if not exists app_trip_rt_wiper_brp_-- (_brp_--     rigel_meta_trip_id string_brp_--     ,wiper_speed double_brp_--     ,wiper_speed_cn string_brp_-- )_brp_-- partitioned by_brp_-- (_brp_--     rigel_meta_event_date string comment '分区'_brp_-- )_brp_-- ;_brp__brp__brp_-- insert overwrite table app_trip_rt_wiper partition (rigel_meta_event_date = '${BIZ_DATE_LINE}')_brp_-- select_brp_--     rigel_meta_trip_id,_brp_--     max(vehicle_detail_info__wiper_speed) as wiper_speed,_brp_--     case   _brp_--         when sum(case when vehicle_detail_info__wiper_speed in (5,6) then 1 else 0 end) > 3

## 字段血缘关系

In [2]:
from sqllineage.runner import LineageRunner
# SUPPORTED_DIALECTS = list(dialect.label for dialect in dialect_readout())
# SUPPORTED_DIALECTS

In [10]:
sql = """
CREATE TABLE if not exists  `dim_rt_trip_odd_info_df`(
  `trip_id` string COMMENT 'trip_id', 
  `trip_odd` string COMMENT 'trip_odd',
  `create_date` string COMMENT 'create_date'
  )
PARTITIONED BY ( 
  `dt` string)
;

WITH app_rt_trip_issue_detail_hf AS (
    SELECT DISTINCT
        trip_id,
        region,
        create_date
    FROM 
        vgds.app_rt_trip_issue_detail_hf
    WHERE 
        dt = '2024-09-11-15'
),

trip_odd_data AS (
    SELECT DISTINCT
        b.trip_id, 
        CASE
            WHEN b.create_date <= a.create_static_date THEN a.trip_odd 
            ELSE 
                CASE
                    WHEN b.region like '%guangzhou%' or b.region like '%广州%' THEN 'ODD2'
                    WHEN b.region like '%beijing_yizhuang%' or b.region like '%北京亦庄%' THEN 'ODD2'
                    WHEN b.region like '%shanghai%' or b.region like '%上海%' THEN 'ODD3'
                    ELSE 'ODD-OTHER'
                END
        END AS trip_odd,
        b.create_date
    FROM 
        vgds.dim_rt_trip_odd_static_df a
    FULL OUTER JOIN 
        app_rt_trip_issue_detail_hf b
    ON a.trip_id = b.trip_id
)

INSERT OVERWRITE TABLE dim_rt_trip_odd_info_df partition (dt = '2024-09-11-15')
select 
    trip_id
    ,trip_odd s
    ,create_date
from
    trip_odd_data;
"""


In [11]:
from sqllineage.runner import LineageRunner
result = LineageRunner(sql,dialect='non-validating')
print(result)

Statements(#): 2
Source Tables:
    vgds.app_rt_trip_issue_detail_hf
    vgds.dim_rt_trip_odd_static_df
Target Tables:
    <default>.dim_rt_trip_odd_info_df



/tmp/ipykernel_1261456/2817020059.py:2: DeprecationWarning: dialect `non-validating` is deprecated, use `ansi` or dialect of your SQL instead. `non-validating` will be completely removed in v1.6.x
  result = LineageRunner(sql,dialect='non-validating')


In [5]:
# result.draw()


## vgds库sql代码查询

In [6]:
sql = get_sql_from_task_id('3100', '31041901')

请求：http://10.88.128.15:8000/dataservice/gateway/v1/api/get_tabel_sql_temp_sql_df，数据：{'dt': '2024-09-13', 'project_id': '3100', 'task_id': '31041901', 'pageSize': 5000, 'page': 1}


In [5]:
def format_sql(sql_str):
    # 替换多个空格为单个空格，避免冗余
    formatted_sql = ' '.join(sql_str.split())

    # 按照指定的符号、关键词等进行换行和格式化
    formatted_sql = formatted_sql.replace('_brp_ _', '\n,') \
                                 .replace('__brp_', ',\n') \
                                 .replace('_brp_', '\n') \
                                 .replace('_ ', ',') \
                                 
     
    # formatted_sql = formatted_sql.replace('--', '\n--')\
                                #  .replace('create table if not exists', '\ncreate table if not exists')\
                                #  .replace('partitioned by', '\npartitioned by')\
                                #  .replace('select', '\nselect')\
                                #  .replace('from', '\nfrom')\
                                #  .replace('where', '\nwhere')\
                                #  .replace('group by', '\ngroup by')\
                                #  .replace('insert overwrite table', '\ninsert overwrite table')\
                                #  .replace('with', '\nwith')\
                                #  .replace('case', '\n    case')\
                                #  .replace('end as', '\n    end as')\
                                #  .replace('when', '\n        when')\
                                #  .replace('else', '\n        else')\

    # 去掉可能多余的空格
    formatted_sql = formatted_sql.replace(' ;', ';')
    # 如果最后不是分号结尾，加上分号
    if formatted_sql[-1] != ';':
        formatted_sql += ';'

    return formatted_sql
    

In [21]:
formatted_sql = format_sql(sql)

result = LineageRunner(formatted_sql,dialect='non-validating')
print(result)

Statements(#): 2
Source Tables:
    vgds.app_rt_trip_issue_detail_hf
    voyager_te_data_platform.auto_labeling_result_dev
Target Tables:
    <default>.app_auto_triage_ml_speed_v1



/tmp/ipykernel_1197320/887143189.py:3: DeprecationWarning: dialect `non-validating` is deprecated, use `ansi` or dialect of your SQL instead. `non-validating` will be completely removed in v1.6.x
  result = LineageRunner(formatted_sql,dialect='non-validating')


In [40]:
import os

def get_source_tables_from_sql(sql,format=False):
    try:
        if format:
            formatted_sql = format_sql(sql)
        else:
            formatted_sql = sql
        result = LineageRunner(formatted_sql,dialect='non-validating')
        source_tables = [str(i).split('.')[1] for i in result.source_tables]
        return source_tables
    except:
        return []

# def get_sql_from_table_name(table_name):
#     query_data = {
#         "table_name": table_name
#     }
#     response = api.post(query_data)
#     return response[0]['file_content']

def get_sql_from_table_name(table_name):
    ## 在file_path内搜索table_name.sql，返回sql
    file_path = '/home/zhengzong/workspace/DS/HiveTrace/HiveTrace/sqllineage/data/vgds/'
    table_name =  table_name+'.sql'
    ## 如果找不到，返回None，否则返回sql
    if table_name not in os.listdir(file_path):
        return None
    with open(file_path+table_name,'r') as f:
        sql = f.read()
    return sql
    

## 给定初始化的sql，获取所有的source tables，返回一个list，继续递归直到无法通过table_name找到sql,
## 把所有的sql都获取到，存在一个list里面，一开始的sql也要加进去
## 再维护一个list，存储已经获取过的table_name，避免重复获取，如果已经获取过，就不再获取
all_sql = []
source_tables = []

def get_all_source_tables(sql,level=3):
    ## 超过5层递归，返回
    if level == 0:
        return
    if sql is None:
        return 
    source_tables_ = get_source_tables_from_sql(sql)
    print(source_tables_)
    for table_name in source_tables_:
        if table_name not in source_tables:
            source_tables.append(table_name)
            sql = get_sql_from_table_name(table_name)
            all_sql.append(sql)
            get_all_source_tables(sql,level-1)
        
            

def combine_sql(sql_list):
    # 去掉None  
    sql_list = [i for i in sql_list if i is not None]
    # print(len(sql_list))
    # 如果最后不是分号结尾，加上分号
    for i in range(len(sql_list)):
        if sql_list[i][-1] != ';':
            sql_list[i] += ';'
    return '\n'.join(sql_list)


all_sql.append(sql)
get_all_source_tables(sql)
print(len(all_sql))

/tmp/ipykernel_1261456/769870701.py:9: DeprecationWarning: dialect `non-validating` is deprecated, use `ansi` or dialect of your SQL instead. `non-validating` will be completely removed in v1.6.x
  result = LineageRunner(formatted_sql,dialect='non-validating')


['app_rt_trip_issue_detail_hf', 'dim_rt_trip_odd_static_df']
['dim_rt_issue_topic_view_hf', 'dim_rt_trip_distance_accumulated_df', 'dim_rt_version_date_range_df', 'dwd_rt3_task_order_package_case_order_hf', 'dwd_rt_issue_with_merged_topic_detail_hf', 'dwd_rt_trip_info_hf', 'dwd_ssevent_data_quality_issue_detail_hf', 'ods_rt_issue_info_1_hf']
['dwd_rt_issue_with_merged_topic_detail_hf']
[]
[]
['case_order']
['dim_rt_trip_station_info', 'dim_rt_trip_weather_di', 'dim_rt_version_date_range_df', 'ds_trip_daily_build', 'ods_daypack_tripstatistics']
['ods_issue_info_1']
['ods_issue_info_1']
17


In [44]:
print(combine_sql(all_sql[0:6]))


CREATE TABLE if not exists  `dim_rt_trip_odd_info_df`(
  `trip_id` string COMMENT 'trip_id', 
  `trip_odd` string COMMENT 'trip_odd',
  `create_date` string COMMENT 'create_date'
  )
PARTITIONED BY ( 
  `dt` string)
;

WITH app_rt_trip_issue_detail_hf AS (
    SELECT DISTINCT
        trip_id,
        region,
        create_date
    FROM 
        vgds.app_rt_trip_issue_detail_hf
    WHERE 
        dt = '2024-09-11-15'
),

trip_odd_data AS (
    SELECT DISTINCT
        b.trip_id, 
        CASE
            WHEN b.create_date <= a.create_static_date THEN a.trip_odd 
            ELSE 
                CASE
                    WHEN b.region like '%guangzhou%' or b.region like '%广州%' THEN 'ODD2'
                    WHEN b.region like '%beijing_yizhuang%' or b.region like '%北京亦庄%' THEN 'ODD2'
                    WHEN b.region like '%shanghai%' or b.region like '%上海%' THEN 'ODD3'
                    ELSE 'ODD-OTHER'
                END
        END AS trip_odd,
        b.create_date
    FROM 
  

In [29]:
"\nCREATE TABLE if not exists  `dim_rt_trip_odd_info_df`(\n  `trip_id` string COMMENT 'trip_id', \n  `trip_odd` string COMMENT 'trip_odd',\n  `create_date` string COMMENT 'create_date'\n  )\nPARTITIONED BY ( \n  `dt` string)\n;\n\nWITH app_rt_trip_issue_detail_hf AS (\n    SELECT DISTINCT\n        trip_id,\n        region,\n        create_date\n    FROM \n        vgds.app_rt_trip_issue_detail_hf\n    WHERE \n        dt = '2024-09-11-15'\n),\n\ntrip_odd_data AS (\n    SELECT DISTINCT\n        b.trip_id, \n        CASE\n            WHEN b.create_date <= a.create_static_date THEN a.trip_odd \n            ELSE \n                CASE\n                    WHEN b.region like '%guangzhou%' or b.region like '%广州%' THEN 'ODD2'\n                    WHEN b.region like '%beijing_yizhuang%' or b.region like '%北京亦庄%' THEN 'ODD2'\n                    WHEN b.region like '%shanghai%' or b.region like '%上海%' THEN 'ODD3'\n                    ELSE 'ODD-OTHER'\n                END\n        END AS trip_odd,\n        b.create_date\n    FROM \n        vgds.dim_rt_trip_odd_static_df a\n    FULL OUTER JOIN \n        app_rt_trip_issue_detail_hf b\n    ON a.trip_id = b.trip_id\n)\n\nINSERT OVERWRITE TABLE dim_rt_trip_odd_info_df partition (dt = '2024-09-11-15')\nselect \n    trip_id\n    ,trip_odd s\n    ,create_date\nfrom\n    trip_odd_data;\n;"

### 本地表导入

In [9]:
import pandas as pd
import numpy as np

sql_code_df = pd.read_csv('/home/zhengzong/workspace/DS/sql任务表查询.csv',engine='python',encoding='gbk')

## 根据file_name筛选
def select_sql_from_file_name(df):
    # 过滤.csv / .xlsx / .xls .zip结尾的
    df = df[~df['file_name'].str.contains('.csv|.xlsx|.xls|.zip')]
    # 过滤Hive2ClickHous、Cooper2Hive、Hive2MySQL、
    df = df[~df['file_name'].str.contains('Hive2ClickHous|Cooper2Hive|Hive2MySQL|MySQL2Hive|MysqlToHive|hive2ck')]
    # 过滤掉test的
    df = df[~df['file_name'].str.contains('test|tmp')]
    # 过滤掉query的
    df = df[~df['file_name'].str.contains('query')]
    # 过滤掉空的file_content
    df = df[df['file_content'].notnull()]
    # 过滤掉.txt/.conf结尾的
    df = df[~df['file_name'].str.contains('.txt|.conf')]
    # 过滤掉datalinkapi_开头的
    df = df[~df['file_name'].str.contains('datalinkapi_')]
    # 过滤掉alter开头的
    df = df[~df['file_name'].str.contains('alter')]
    return df

sql_code_df = select_sql_from_file_name(sql_code_df)

In [7]:
sql_code_df[sql_code_df['file_name'] == 'app_auto_triage_ml_speed']['file_content'].iloc[0]

"--SPARK_SQL_brp_--********************************************************************--_brp_--author:jimmylimao_brp_--create time:2024-04-12 16:56:27_brp_--desc:Auto triage demo data_brp_--remind:请在资源引用中添加需要引用的资源_brp_--********************************************************************--_brp_create table if not exists app_auto_triage_ml_speed_v1(_brp_    issue_id string__brp_    if_tags_unstable string__brp_    if_ub_offline string__brp_    if_disengage string__brp_    trip_id string__brp_    disengage_time bigint__brp_    timestamp string__brp_    speed string__brp_    maneuver string__brp_    planning_signal string__brp_    speed_discomfort string__brp_    trajectory_guider_limit_min string__brp_    reference_profile_speed string__brp_    trajectory_jerk string_brp_);_brp_insert overwrite table app_auto_triage_ml_speed_v1_brp_select_brp_    b.issue_id__brp_    if(tags_ops like '%不平顺%' _ 'yes'_ 'no') if_tags_unstable__brp_    if(channel = 'OFFLINE'_ 'yes'_ 'no') if_ub_offline__brp_

In [15]:
sql = sql_code_df[sql_code_df['file_name'] == 'app_auto_triage_ml_speed']['file_content'].iloc[0]

In [16]:
sql

"--SPARK_SQL_brp_--********************************************************************--_brp_--author:jimmylimao_brp_--create time:2024-04-12 16:56:27_brp_--desc:Auto triage demo data_brp_--remind:请在资源引用中添加需要引用的资源_brp_--********************************************************************--_brp_create table if not exists app_auto_triage_ml_speed_v1(_brp_    issue_id string__brp_    if_tags_unstable string__brp_    if_ub_offline string__brp_    if_disengage string__brp_    trip_id string__brp_    disengage_time bigint__brp_    timestamp string__brp_    speed string__brp_    maneuver string__brp_    planning_signal string__brp_    speed_discomfort string__brp_    trajectory_guider_limit_min string__brp_    reference_profile_speed string__brp_    trajectory_jerk string_brp_);_brp_insert overwrite table app_auto_triage_ml_speed_v1_brp_select_brp_    b.issue_id__brp_    if(tags_ops like '%不平顺%' _ 'yes'_ 'no') if_tags_unstable__brp_    if(channel = 'OFFLINE'_ 'yes'_ 'no') if_ub_offline__brp_

In [17]:
print(format_sql(sql))

--SPARK_SQL
--********************************************************************--
--author:jimmylimao
--create time:2024-04-12 16:56:27
--desc:Auto triage demo data
--remind:请在资源引用中添加需要引用的资源
--********************************************************************--
create table if not exists app_auto_triage_ml_speed_v1(
 issue_id string,
 if_tags_unstable string,
 if_ub_offline string,
 if_disengage string,
 trip_id string,
 disengage_time bigint,
 timestamp string,
 speed string,
 maneuver string,
 planning_signal string,
 speed_discomfort string,
 trajectory_guider_limit_min string,
 reference_profile_speed string,
 trajectory_jerk string
);
insert overwrite table app_auto_triage_ml_speed_v1
select
 b.issue_id,
 if(tags_ops like '%不平顺%' ,'yes','no') if_tags_unstable,
 if(channel = 'OFFLINE','yes','no') if_ub_offline,
 if(found_problem_category = '接管','yes','no') if_disengage,
 a.trip_id,
 disengage_time,
 `timestamp`,
 speed,
 maneuver,
 planning_signal,
 speed_discomfort,
 trajecto

In [18]:
from tqdm import tqdm
import os
import shutil

data_path = '/home/zhengzong/workspace/DS/HiveTrace/HiveTrace/sqllineage/data/vgds/'

## 清空data_path下的文件
shutil.rmtree(data_path)
os.makedirs(data_path)


## 遍历sql_code_df,将file_content写入data_path, 并以file_name命名
for index, row in tqdm(sql_code_df.iterrows()):
    file_name = row['file_name'] + '.sql'
    file_content = row['file_content']
    ## 格式化sql
    print(file_name)
    try:
        formatted_sql = format_sql(file_content)
    except:
        print(file_name, '格式化失败')
        print(file_content)
        continue
    
    with open(data_path + file_name, 'w') as f:
        f.write(formatted_sql)

0it [00:00, ?it/s]

510it [00:00, 5096.83it/s]

dwd_autolabel_tp_lane_change_july.sql
get_latlon_by_pose.py.sql
route_grade_odd.sql
ego_lean_one_side_2.sql
dwd_autolabel_junction.sql
dwd_weather_district.sql
dwm_autolabel_labeled_detail.sql
app_temp_dwm_car_eff.sql
pull_status_calculate.sql
app_rt_followcar_coverage_inserthistorydata.sql
app_rt_followcar_coverage_df.sql
dm_scenario_lane_change_d.sql
dwd_asm_ego_obj_contour.sql
app_autotopic_issue_diff_detail.sql
app_ds_autotopic_accuracy.sql
dwd_release_binary.sql
dwm_voyager_car_group_on_off_line.sql
app_ds_get_latlon_by_pose.sql
holiday_for_ops.sql
dwd_autolabel_road_info.sql
dwd_sim_datasim_scenario_detail.sql
app_dwd_car_eff_temp.sql
ops_team_leader.sql
app_dwd_people_eff_temp.sql
app_dwm_car_eff_temp.sql
dim_ds_issue_pose_di.sql
dim_ds_issue_speed_di.sql
dwd_ds_mdbi_issue_new_di.sql
app_ds_mdbi_trip_issue_v4_di.sql
dwd_autolabel_rule_lane_change.sql
ce_issue_cretieria_to_hive.sql
dwd_voyager_opstrain.sql
dwd_ds_trip_exemption_st_di.sql
app_ds_mdbi_trip_issue_v3_di.sql
dwd_rtmap

1068it [00:00, 5042.96it/s]

real_time_issue_detail.sql
update_order_vip_field.sql
junction_num.sql
lane_change_num.sql
nudge_num.sql
planning_lane_change.sql
remote_assist.sql
segment_crosswalk.sql
segment_uturn.sql
tp_crossing.sql
tp_cut_in.sql
tp_cyc_in_regular.sql
tp_drive_wrong_direction.sql
tp_jaywalking.sql
tp_moving_against.sql
tp_squeeze_lane.sql
traffic_participants.sql
app_order_execute_status_df.sql
app_ds_fallback_planning_issue.sql
brake_collect.sql
simulation_metrics_result_hive_ot.sql
app_loc_recall_summary.sql
app_cr_build_relation.sql
app_ds_asm_fallback_smbk_hdbk.sql
dwd_trip_ctz_ego_mode_issue_df_baseline_2023apr_repair.sql
app_msg_drop_trip.sql
dwd_map_draft_data_and_spaceway_task_association_hf.sql
dws_middleware_object.sql
dws_middleware_object_day.sql
app_rt_middleware_latency_hf.sql
adjusted_play_cnt.sql
dwd_rtmap_spaceway_task_and_fixed_point_df.sql
dim_rt_trip_bad_case_congestion_finished_df.sql
app_trip_car_control_event_order_flow_df.sql
app_trip_rt_event_mrc_di_ot.sql
dwd_ra_intervent